<p style="text-align: center">
<img src="../../../assets/images/dtlogo.png" alt="Duckietown" width="50%">
</p>


# Training Your Model

You'll train a YOLOv5 nano model on the dataset prepared in the [previous notebook](../02-Setup-Data-Collection/setup.ipynb).

YOLOv5 training needs a GPU. The recommended path is **Google Colab** — free GPU runtime, no local setup. If you have a CUDA-capable GPU, you can also train locally (see the optional section at the bottom).


## 1. Package the dataset

We zip the `train/` and `val/` folders so you can upload the archive to Google Drive in one step.


In [1]:
import sys
import os

notebook_dir = os.path.abspath('')
project_root = os.path.abspath(os.path.join(notebook_dir, '..', '..', '..', '..'))

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f'Project root: {project_root}')


Project root: c:\Users\user\Desktop\studies 5\software engineering practical course\KvatiTown


In [2]:
%load_ext autoreload
%autoreload 2


Failed to read module file 'C:\Users\user\AppData\Local\Programs\Python\Python311\Lib\pydoc_data\topics.py' for module 'pydoc_data.topics': UnicodeDecodeError
Traceback (most recent call last):
  File "c:\Users\user\Desktop\studies 5\software engineering practical course\KvatiTown\.venv\Lib\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\user\Desktop\studies 5\software engineering practical course\KvatiTown\.venv\Lib\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\user\AppData\Local\Programs\Python\Python311\Lib\importlib\__init__.py", line 126, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1204, in _gcd_impor

In [6]:
import shutil

DATASET_DIR = os.path.join(project_root, "tasks", "object_detection", "dataset")
out_basename = os.path.join(project_root, "tasks", "object_detection", "object_detection_dataset")
out_full = f"{out_basename}.zip"

if os.path.exists(out_full):
    print(f"Archive already exists at: {out_full}")
    print("Delete it manually if you want to regenerate.")
else:
    import tempfile
    with tempfile.TemporaryDirectory() as tmp:
        for sub in ["train", "val"]:
            src = os.path.join(DATASET_DIR, sub)
            if os.path.isdir(src):
                shutil.copytree(src, os.path.join(tmp, sub))
        shutil.make_archive(out_basename, "zip", tmp)
        print(f"Archive created: {out_full}")


Archive created: c:\Users\user\Desktop\studies 5\software engineering practical course\KvatiTown\tasks\object_detection\object_detection_dataset.zip


## 2. Train on Google Colab

Upload the zip from above to your Google Drive, then open a Colab notebook with GPU runtime (`Runtime > Change runtime type > T4 GPU`).

Paste the following into a Colab cell and run it. It clones YOLOv5, unpacks your dataset, writes the data config, and starts training.

```python
# Cell 1 - pin a torch version that works with YOLOv5 v7.0
!pip install -q "torch<2.6" "torchvision<0.21"

# Cell 2 - disable W&B prompt (optional logging service)
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"]     = "disabled"

# Cell 3 - mount drive and unpack the dataset
from google.colab import drive
drive.mount('/content/drive')

!cp /content/drive/MyDrive/object_detection_dataset.zip /content/
!unzip -q /content/object_detection_dataset.zip -d /content/dataset

# Cell 4 - clone YOLOv5 and install its requirements
!git clone https://github.com/ultralytics/yolov5.git /content/yolov5
%cd /content/yolov5
!pip install -q -r requirements.txt

# Cell 5 - write the data config
data_yaml = """
  train: /content/dataset/train/images
  val:   /content/dataset/val/images
  nc: 3
  names: ['duckie', 'truck', 'sign']
"""
with open('/content/yolov5/duckietown.yaml', 'w') as f:
    f.write(data_yaml)

# Cell 6 - train
!python train.py --img 416 --batch 32 --epochs 100 \
    --data duckietown.yaml --weights yolov5n.pt \
    --cfg ./models/yolov5n.yaml

# Cell 7 - save the model with onnx format
!python export.py --weights runs/train/exp/weights/best.pt --include onnx --opset 10 --simplify
```

Training will take 10–30 minutes. When it finishes, find the best weights at:

```
/content/yolov5/runs/train/exp/weights/best.onnx
```

Download `best.onnx` to your local machine.


## 3. Place the model locally

Move the downloaded `best.onnx` into the project under:

```
tasks/object_detection/models/best.onnx
```

(Create the `models/` folder if it doesn't exist.)

The path is read by `MODEL_PATH` in [`integration_activity.py`](../../packages/integration_activity.py). If you want to use a different filename or location, edit that variable.


## Optional - train locally

If you have a CUDA GPU, you can skip Colab. Run the same commands as above in a local terminal, after activating your virtual environment with the optional `object_detection` extras installed:

```bash
pip install ".[object_detection]"
git clone https://github.com/ultralytics/yolov5.git
cd yolov5
pip install -r requirements.txt
# write duckietown.yaml as shown above, pointing at your local dataset
python train.py --img 416 --batch 16 --epochs 100 \
    --data duckietown.yaml --weights yolov5n.pt
```

If you hit `_pickle.UnpicklingError: Weights only load failed`, downgrade torch:

```bash
pip install "torch<2.6" "torchvision<0.21"
```


## Next step

Continue with the [Integration notebook](../04-Integration/integration.ipynb) to run your trained model on the live camera feed.
